# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 05 · Vector Database

This notebook demonstrates the **Vector Database** stage of the RAG pipeline.

A vector database is used to store the embeddings generated from the document chunks, together with their associated metadata. It provides efficient similarity search over the embedding space, allowing the RAG system to retrieve the document chunks that are most relevant to a user's query. During retrieval, the query is also converted into an embedding, and the vector database identifies the stored embeddings that are closest to it according to a similarity measure.

For this project, **ChromaDB** was selected as the vector database. Chroma provides a simple Python interface for storing and querying embeddings and integrates naturally with the **LangChain** components used throughout the project. It also supports both **persistent storage** and **in-memory databases**, which makes it suitable for both local development and the deployment requirements of the *Streamlit* application.

The `VectorDatabase` class is defined in `src/github_rag/vectordatabase/chroma_store.py`.

**Notebook objectives**

* Create a local, persistent Chroma database stored on disk.
* Create an in-memory Chroma database.
* Store document embeddings and their associated metadata.
* Perform simple similarity retrieval tasks to understand how ChromaDB works in the context of a RAG system.

## Imports and Configuration

In [2]:
import time
from pathlib import Path
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model
from github_rag.vectordatabase.chroma_store import ChromaStore

You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects, which combines XGBoost and SHAP to build a predictive and explainable system for classifying urban typologies.

In [3]:
REPOSITORY_URL = "https://github.com/davidfernxndez/urban-typology-xai"

## 1. Ingestion -> processing -> chunking

This section apply the Ingestion, processing and chunking phase to get the prepared *LangChain Documents* for Embedding and VectorDB storing.

In [4]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

Number of documents ingested: 28


In [5]:
# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents_processed = processor.process(documents)

print(f"Number of documents before processing: {len(documents_processed)}")

Number of documents before processing: 324


In [6]:
# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

# Apply chunking
documents_chunking = chunker.chunk(documents_processed)

print(f"Number of documents before chunking: {len(documents_chunking)}")

Number of documents before chunking: 985


## 2. Load the Embedding model

We first need to load the embedding model that will be passed to the ChromaDB instance. The same model used to generate the document embeddings must be used to embed user queries during the retrieval process.

In [7]:
# Set the HuggingFace embedding model name
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Load the model
embedding_model = create_embedding_model(embedding_model_name)

## 3. Create Chroma Database

The `ChromaStore` class supports two different storage modes depending on the execution environment. For local development, it can create a **persistent Chroma database** stored on disk. The database is stored in the `/data` directory at the root of the repository, allowing it to persist between executions and making it convenient for development and experimentation.

It also supports creating an **in-memory Chroma database**, which is intended for the Streamlit deployment. In this mode, the vector database exists only during the current application session and does not require persistent storage on disk.

### 3.1 Persistent Chroma database

In [8]:
start_time = time.time()
# Create ChromaDB instance for local disk storing
vectorDB_local = ChromaStore(
    mode="local",
    collection_name="david_repo"
)

# Create the database with documents and the embedding model
vector_store = vectorDB_local.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)
print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vector_store._collection.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

Database generation time: 19.67 seconds
Documents provided: 985
Documents stored:   985


We can verify that it has been created on disk.

In [10]:
persist_directory = Path("../data/chroma")

print("Database exists:", persist_directory.exists())
print("Files:")
for path in persist_directory.iterdir():
    print(path)

Database exists: True
Files:
..\data\chroma\60ef5686-1379-40e7-a686-e64336fc8672
..\data\chroma\chroma.sqlite3


### 3.1 In-Memory Chroma database

In [11]:
start_time = time.time()
# Create ChromaDB instance using in-memory storage
vectorDB_memory = ChromaStore(
    mode="memory",
    collection_name="david_repo_memory"
)

# Create the database
vector_store_memory = vectorDB_memory.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)

print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vector_store_memory._collection.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

Database generation time: 21.02 seconds
Documents provided: 985
Documents stored:   985


The following demonstrates that the database doesn`t exist in a local directory.

In [12]:
# Verify that no persistence directory is configured
print(f"Persistence directory: {vectorDB_memory.persist_directory}")

Persistence directory: None


## 4. Perform Retrieval from ChromaDB

ChromaDB provides built-in methods for retrieving documents based on the similarity between a query and the stored embeddings. These methods allow us to perform semantic searches over the vector database and retrieve the document chunks that are most relevant to a given query.

In this section, we perform some simple retrieval examples to verify that the documents stored in ChromaDB can be successfully retrieved based on their semantic similarity to a user query.

The get method allows us to retrieve some documents and check their ID, metadata and text content.

In [16]:
results = vector_store._collection.get(limit=3)

for i in range(3):
    print(f"Document {i}")
    print("ID:", results["ids"][i])
    print("Metadata:", results["metadatas"][i])
    print("Content:", results["documents"][i][:500])
    print("-" * 80)
    print("\n")

Document 0
ID: 9db6c62c-3732-4b3a-80e2-1f8c0739d2e6
Metadata: {'file_type': '.toml', 'file_path': 'pyproject.toml', 'repository': 'davidfernxndez/urban-typology-xai', 'source': 'https://github.com/davidfernxndez/urban-typology-xai/blob/main/pyproject.toml', 'branch': 'main'}
Content: [build-system]
requires = ["setuptools>=61.0.0"]
build-backend = "setuptools.build_meta"

[project]
name = "urban-typology-xai"
version = "0.1.0"
description = "Explainable machine learning for the classification of residential enclosure patterns"
readme = "README.md"
requires-python = ">=3.11"

[tool.setuptools]
packages = ["src"]
--------------------------------------------------------------------------------


Document 1
ID: bc85dcce-617a-4cba-807a-fa39d6e6257b
Metadata: {'file_path': 'README.md', 'header_3': 'A Comparative Analysis of the Performance and Interpretability of Supervised Learning Models for the Classification of Residential Enclosure Patterns', 'file_type': '.md', 'repository': 'davidfern

We can do a simple RAG retrieval example using the `similarity_search_with_score` method.

In [20]:
# Verify semantic search
query = "¿How is cost-sensitive learning implemented in XGBoost multiclass problem?"

results = vector_store.similarity_search_with_score(
    query,
    k=5
)

print(f"\nQuery: {query}\n")

for i, (document, distance) in enumerate(results, start=1):
    print(f"Result {i}")
    print(f"Distance: {distance:.4f}")
    print(f"Source: {document.metadata.get('source')}")
    print(f"File: {document.metadata.get('file_path')}")
    print(document.page_content)
    print("-" * 80)


Query: ¿How is cost-sensitive learning implemented in XGBoost multiclass problem?

Result 1
Distance: 7.7049
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/README.md
File: README.md
| `balanced_xgb.py`        | Custom wrapper implementing cost-sensitive XGBoost for multiclass classification.              |
| `production_training.py` | Functions for training and optimizing the selected model using the complete available dataset. |
| `XAI_utils.py`           | Functions supporting global and local model interpretability analyses.                         |
--------------------------------------------------------------------------------
Result 2
Distance: 10.1323
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/notebooks/3.1_Performance_methodology.ipynb
File: notebooks/3.1_Performance_methodology.ipynb
El modelo *XGBoost* se ha implementado mediante la librería *xgboost* desarrollada por los propios autores del algoritmo. A diferencia de los 